# The clustering script that ran out of memory at 100 000

A pairwise-distance matrix has $n^2$ cells. Ten times the data is a hundred times
the memory, and an allocation that does not fit never starts.

In [1]:
import numpy as np

for n in [1_000, 10_000, 100_000, 1_000_000]:
    print(f"n = {n:>9,}: matrix {8 * n * n / 1e9:>12,.1f} GB    the data itself (12 columns) {8 * n * 12 / 1e6:>8,.1f} MB")

n =     1,000: matrix          0.0 GB    the data itself (12 columns)      0.1 MB
n =    10,000: matrix          0.8 GB    the data itself (12 columns)      1.0 MB
n =   100,000: matrix         80.0 GB    the data itself (12 columns)      9.6 MB
n = 1,000,000: matrix      8,000.0 GB    the data itself (12 columns)     96.0 MB


## See it happen (safely)

Allocate the matrix for a small $n$ and read its size; the failure at $10^5$ is
what the error message in the lecture shows, so we stop at $10^4$.

In [2]:
from scipy.spatial.distance import cdist

for n in [1_000, 3_000, 10_000]:
    X = np.random.default_rng(0).normal(size=(n, 12))
    D = cdist(X, X)                       # the full n x n matrix
    print(f"n = {n:>6}: {D.nbytes / 1e6:8.1f} MB")

n =   1000:      8.0 MB
n =   3000:     72.0 MB


n =  10000:    800.0 MB


## The escape: never form the matrix

K-means compares each point with $K$ centers, not with every other point:
$O(nK)$ memory per step instead of $O(n^2)$.

In [3]:
def kmeans_step(X, centers):
    d = cdist(X, centers)                # n x K, not n x n
    labels = d.argmin(axis=1)
    return np.array([X[labels == k].mean(axis=0) for k in range(len(centers))]), d.nbytes

X = np.random.default_rng(0).normal(size=(100_000, 12))
centers, nbytes = kmeans_step(X, X[:5])
print(f"n = 100,000, K = 5: {nbytes / 1e6:.1f} MB for the distances instead of 80,000 MB")

n = 100,000, K = 5: 4.0 MB for the distances instead of 80,000 MB


**What we used:** memory complexity as a growth rate you can read off the code
before running it (session 8, sections 2, 8 and 9); K-means is session 10.